# Passo 1: Ingestão dos dados

A partir do ODS baixado do site de dados
[https://dados.gov.br/dados/conjuntos-dados/filmes-e-sessoes-da-programadora-brasil]

In [1]:
!pip install odfpy

In [2]:
import pandas as pd

# Caminho do arquivo (certifique-se de que o arquivo está na mesma pasta do notebook)
caminho_arquivo = 'filmes.ods'

# Carregar a planilha
# Se o arquivo tiver várias abas, você pode especificar o nome com sheet_name='NomeDaAba'
df = pd.read_excel(caminho_arquivo, engine='odf')

# Visualizar as primeiras linhas
print("Primeiras linhas do arquivo:")
display(df.head())

# # Verificar informações das colunas
# print("\nEstrutura dos dados:")
# df.info()

Primeiras linhas do arquivo:


,codigo,status,data_cadastro,data_alteracao,titulo,titulo_en,sinopse,sinopse_editada,sinopse_en,diretor,...,dist_exclusivo,pat_minc,pat_minc_edital,pat_minc_contrato,desc_classificacao,autor_da_trilha,animador,dublagem,cod_inscrito_pb,infanto_juvenil
0,148,1,2008-03-01 00:00:00,2011-08-17 11:14:10,Amarelo manga,Mango Yellow,"Guiados pela paixão, os personagens de Amarelo...","Guiados pela paixão, os personagens de Amarelo...",NaN,Cl&aacute;udio Assis,...,N,N,BO - Edital de concurso nº 2 de 27 de abril de...,4412000,NaN,Jorge du Peixe e Lúcio Maia,NaN,NaN,1,N
1,149,1,2008-03-01 00:00:00,2011-08-16 19:00:13,A canga,NaN,"Num descampado, no meio de uma lavoura seca, u...","Num descampado, no meio de uma lavoura seca, u...",NaN,Marcus Vilar,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N
2,150,1,2008-03-01 00:00:00,2012-09-28 15:41:32,"Batismo de Carmencita, 25 de junho de 1921",NaN,Um dos assuntos de um cinejornal. Cerimônia de...,Um dos assuntos de um cinejornal. Cerimônia de...,NaN,Autoria desconhecida,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N
3,151,1,2008-03-01 00:00:00,2012-09-28 15:49:00,Exemplo regenerador,NaN,Um marido farrista deixa a esposa só em casa n...,Um marido farrista deixa a esposa só em casa n...,NaN,Jos&eacute; Medina,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N
4,152,1,2008-03-01 00:00:00,2012-10-09 15:20:31,Brasilianas: Aboio e Cantigas,NaN,O Canto utilizado pelo vaqueiro para reunir a ...,O Canto utilizado pelo vaqueiro para reunir a ...,NaN,Humberto Mauro,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N


# Limpeza

## 1. Selecionar colunas de interesse

In [3]:
colunas_interesse = [
    "titulo",     # ajuste para o nome de coluna real
    "data_cadastro",
    "diretor",
    "sinopse",
]
filmes_cols = df[colunas_interesse].copy()
filmes_cols

,titulo,data_cadastro,diretor,sinopse
0,Amarelo manga,2008-03-01 00:00:00,Cl&aacute;udio Assis,"Guiados pela paixão, os personagens de Amarelo..."
1,A canga,2008-03-01 00:00:00,Marcus Vilar,"Num descampado, no meio de uma lavoura seca, u..."
2,"Batismo de Carmencita, 25 de junho de 1921",2008-03-01 00:00:00,Autoria desconhecida,Um dos assuntos de um cinejornal. Cerimônia de...
3,Exemplo regenerador,2008-03-01 00:00:00,Jos&eacute; Medina,Um marido farrista deixa a esposa só em casa n...
4,Brasilianas: Aboio e Cantigas,2008-03-01 00:00:00,Humberto Mauro,O Canto utilizado pelo vaqueiro para reunir a ...
...,...,...,...,...
1874,Realejo,2012-10-15 16:39:44,-,"A cada lua cheia, um realejo mágico decide o f..."
1875,Regando bigodes,2012-10-15 16:41:59,-,"Felipe, sete anos, sonha em ter bigodes como o..."
1876,O reino do chocolate,2012-10-15 16:43:25,-,A história de um planeta onde pessoas de cores...
1877,Sonho de menina,2012-10-15 16:45:36,-,Os sonhos de uma menina pobre e o acaso mudand...


## 2. Padronizar textos

In [4]:
# Padronizar título para facilitar
filmes_cols["titulo_limpo"] = (
    filmes_cols["titulo"]
    .str.strip() # remove espaços extras antes ou depois do titulo
)

In [5]:
# Remover acentos, pontuação, etc. para facilitar o matching
import unicodedata
import re
import html

def normalizar(t):
    if pd.isna(t):
        return t
    t = html.unescape(t)
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    t = t.upper() # colocar em maiúsculas
    t = re.sub(r"[^A-Z0-9 ]", " ", t) # trocar tudo que não é letra ou número por espaço
    t = re.sub(r"\s+", " ", t).strip() # remover múltiplos espaços por apenas um
    return t

filmes_cols["diretor_norm"] = filmes_cols["diretor"].apply(normalizar)
filmes_cols["titulo_norm"] = filmes_cols["titulo_limpo"].apply(normalizar)
filmes_cols = filmes_cols.drop(columns=["titulo_limpo", "titulo", "diretor"])
filmes_cols

,data_cadastro,sinopse,diretor_norm,titulo_norm
0,2008-03-01 00:00:00,"Guiados pela paixão, os personagens de Amarelo...",CLAUDIO ASSIS,AMARELO MANGA
1,2008-03-01 00:00:00,"Num descampado, no meio de uma lavoura seca, u...",MARCUS VILAR,A CANGA
2,2008-03-01 00:00:00,Um dos assuntos de um cinejornal. Cerimônia de...,AUTORIA DESCONHECIDA,BATISMO DE CARMENCITA 25 DE JUNHO DE 1921
3,2008-03-01 00:00:00,Um marido farrista deixa a esposa só em casa n...,JOSE MEDINA,EXEMPLO REGENERADOR
4,2008-03-01 00:00:00,O Canto utilizado pelo vaqueiro para reunir a ...,HUMBERTO MAURO,BRASILIANAS ABOIO E CANTIGAS
...,...,...,...,...
1874,2012-10-15 16:39:44,"A cada lua cheia, um realejo mágico decide o f...",,REALEJO
1875,2012-10-15 16:41:59,"Felipe, sete anos, sonha em ter bigodes como o...",,REGANDO BIGODES
1876,2012-10-15 16:43:25,A história de um planeta onde pessoas de cores...,,O REINO DO CHOCOLATE
1877,2012-10-15 16:45:36,Os sonhos de uma menina pobre e o acaso mudand...,,SONHO DE MENINA


## 3. Remover linhas com dados incompletos 

In [6]:
# Lista de colunas para verificar
colunas = ['data_cadastro', 'sinopse', 'diretor_norm', 'titulo_norm']

# Filtra linhas onde qualquer uma das colunas está nula ou é uma string vazia
filtro_vazios = filmes_cols[
    filmes_cols[colunas].isnull().any(axis=1) | 
    (filmes_cols[colunas] == "").any(axis=1)
]
filtro_vazios

,data_cadastro,sinopse,diretor_norm,titulo_norm
247,2008-04-25 14:40:18,Documentário com atmosfera noir. 33 anos é a i...,KIKO GOIFMAN,NaN
450,2008-08-20 14:45:32,Um pouco das inusitadas reações dos alunos de ...,GUIWHI SANTOS,NaN
683,2009-03-12 15:06:30,NaN,SERGIO SANZ,SOLDADO DE DEUS
688,2009-03-16 19:57:14,NaN,JOM TOB AZULAY,EUPHRASIA
689,2009-03-16 20:37:32,NaN,JOM TOB AZULAY,ILHA GRANDE
...,...,...,...,...
1874,2012-10-15 16:39:44,"A cada lua cheia, um realejo mágico decide o f...",,REALEJO
1875,2012-10-15 16:41:59,"Felipe, sete anos, sonha em ter bigodes como o...",,REGANDO BIGODES
1876,2012-10-15 16:43:25,A história de um planeta onde pessoas de cores...,,O REINO DO CHOCOLATE
1877,2012-10-15 16:45:36,Os sonhos de uma menina pobre e o acaso mudand...,,SONHO DE MENINA


In [7]:
# o pandas funciona melhor com NaN (nulo) do que um string vazio. Vamos substituir para facilitar.
import numpy as np

# Transforma "" em NaN
filmes_cols.replace("", np.nan, inplace=True)

# Agora basta usar o isna() ou isnull()
linhas_com_erro = filmes_cols[filmes_cols[colunas].isnull().any(axis=1)]
linhas_com_erro

,data_cadastro,sinopse,diretor_norm,titulo_norm
247,2008-04-25 14:40:18,Documentário com atmosfera noir. 33 anos é a i...,KIKO GOIFMAN,NaN
450,2008-08-20 14:45:32,Um pouco das inusitadas reações dos alunos de ...,GUIWHI SANTOS,NaN
683,2009-03-12 15:06:30,NaN,SERGIO SANZ,SOLDADO DE DEUS
688,2009-03-16 19:57:14,NaN,JOM TOB AZULAY,EUPHRASIA
689,2009-03-16 20:37:32,NaN,JOM TOB AZULAY,ILHA GRANDE
...,...,...,...,...
1874,2012-10-15 16:39:44,"A cada lua cheia, um realejo mágico decide o f...",NaN,REALEJO
1875,2012-10-15 16:41:59,"Felipe, sete anos, sonha em ter bigodes como o...",NaN,REGANDO BIGODES
1876,2012-10-15 16:43:25,A história de um planeta onde pessoas de cores...,NaN,O REINO DO CHOCOLATE
1877,2012-10-15 16:45:36,Os sonhos de uma menina pobre e o acaso mudand...,NaN,SONHO DE MENINA


In [8]:
# Vamos ver o tamanho do estrago, tem muita coisa faltando?

# Mostra a soma de valores NaN por coluna
print(filmes_cols[colunas].isnull().sum())

data_cadastro      0
sinopse           70
diretor_norm     312
titulo_norm        2
dtype: int64


Vamos deixar as linhas com sinopse vazia, mas remover as que não tem diretor.

In [9]:
# Removendo as linhas onde 'diretor_norm' é NaN
# subset garante que apenas a falta de diretor cause a exclusão
filmes_cols.dropna(subset=['diretor_norm', 'titulo_norm'], inplace=True)

# 5. Resetando o índice para manter a organização sequencial (0, 1, 2...)
filmes_cols.reset_index(drop=True, inplace=True)
filmes_cols

,data_cadastro,sinopse,diretor_norm,titulo_norm
0,2008-03-01 00:00:00,"Guiados pela paixão, os personagens de Amarelo...",CLAUDIO ASSIS,AMARELO MANGA
1,2008-03-01 00:00:00,"Num descampado, no meio de uma lavoura seca, u...",MARCUS VILAR,A CANGA
2,2008-03-01 00:00:00,Um dos assuntos de um cinejornal. Cerimônia de...,AUTORIA DESCONHECIDA,BATISMO DE CARMENCITA 25 DE JUNHO DE 1921
3,2008-03-01 00:00:00,Um marido farrista deixa a esposa só em casa n...,JOSE MEDINA,EXEMPLO REGENERADOR
4,2008-03-01 00:00:00,O Canto utilizado pelo vaqueiro para reunir a ...,HUMBERTO MAURO,BRASILIANAS ABOIO E CANTIGAS
...,...,...,...,...
1560,2012-09-13 16:26:11,"Jovem engenheiro, Cristóvão conhece Joana, uma...",MIGUEL RAMOS,UM DOIS TRES VULCAO
1561,2012-09-27 11:36:33,NaN,ALEX TEIX RICARDO GELAIN E ROSARIO BOYER,BATUTA O RATINHO AVENTUREIRO
1562,2012-10-15 12:19:16,Férias de verão no interior. Igor tem quatro a...,MARILIA NOGUEIRA,ANTES QUE O VERAO ACABE
1563,2012-10-15 12:33:27,O retrato da cidade de São Paulo a partir de a...,ALOYSIO RAULINO,LACRIMOSA


In [10]:
# Funcionou?

# Mostra a soma de valores NaN por coluna
print(filmes_cols[colunas].isnull().sum())

data_cadastro     0
sinopse          31
diretor_norm      0
titulo_norm       0
dtype: int64


## 4. Tratar o tipo das colunas

In [11]:
# Verifica o tipo de cada coluna
filmes_cols.dtypes

data_cadastro    str
sinopse          str
diretor_norm     str
titulo_norm      str
dtype: object

Podemos ver que a data_cadastro não veio como *datetime*, e sim como *string*.

In [12]:
# Converte a coluna para datetime
filmes_cols['data_cadastro'] = pd.to_datetime(filmes_cols['data_cadastro'], errors='coerce')

# O 'errors=coerce' transformará qualquer data inválida em NaT (Not a Time), 
# que funciona como o NaN para datas.

In [13]:
# Verifica o tipo de cada coluna
filmes_cols.dtypes

data_cadastro    datetime64[us]
sinopse                     str
diretor_norm                str
titulo_norm                 str
dtype: object

In [14]:
filmes_cols['data_cadastro'].dt.year # apenas funciona em colunas do tipo "date"

0       2008
1       2008
2       2008
3       2008
4       2008
        ... 
1560    2012
1561    2012
1562    2012
1563    2012
1564    2012
Name: data_cadastro, Length: 1565, dtype: int32

# Transformação

Obtemos a década a partir do ano.

In [15]:
filmes_cols['decada'] = (filmes_cols['data_cadastro'].dt.year // 10) * 10
filmes_cols

,data_cadastro,sinopse,diretor_norm,titulo_norm,decada
0,2008-03-01 00:00:00,"Guiados pela paixão, os personagens de Amarelo...",CLAUDIO ASSIS,AMARELO MANGA,2000
1,2008-03-01 00:00:00,"Num descampado, no meio de uma lavoura seca, u...",MARCUS VILAR,A CANGA,2000
2,2008-03-01 00:00:00,Um dos assuntos de um cinejornal. Cerimônia de...,AUTORIA DESCONHECIDA,BATISMO DE CARMENCITA 25 DE JUNHO DE 1921,2000
3,2008-03-01 00:00:00,Um marido farrista deixa a esposa só em casa n...,JOSE MEDINA,EXEMPLO REGENERADOR,2000
4,2008-03-01 00:00:00,O Canto utilizado pelo vaqueiro para reunir a ...,HUMBERTO MAURO,BRASILIANAS ABOIO E CANTIGAS,2000
...,...,...,...,...,...
1560,2012-09-13 16:26:11,"Jovem engenheiro, Cristóvão conhece Joana, uma...",MIGUEL RAMOS,UM DOIS TRES VULCAO,2010
1561,2012-09-27 11:36:33,NaN,ALEX TEIX RICARDO GELAIN E ROSARIO BOYER,BATUTA O RATINHO AVENTUREIRO,2010
1562,2012-10-15 12:19:16,Férias de verão no interior. Igor tem quatro a...,MARILIA NOGUEIRA,ANTES QUE O VERAO ACABE,2010
1563,2012-10-15 12:33:27,O retrato da cidade de São Paulo a partir de a...,ALOYSIO RAULINO,LACRIMOSA,2010


# Visualização

Agora, contar a quantidade por década:

In [16]:
filmes_cols["decada"].value_counts().sort_index()

decada
2000    879
2010    686
Name: count, dtype: int64

Ou contagem de filme por diretor

In [17]:
# Contar a quantidade de filmes por diretor
ranking_diretores = filmes_cols['diretor_norm'].value_counts()

# Exibir os 10 diretores com mais filmes
print("Top 10 Diretores com mais filmes:")
ranking_diretores.head(10)

Top 10 Diretores com mais filmes:


diretor_norm
JOM TOB AZULAY             14
SERGIO PEO                 10
JOAO BATISTA DE ANDRADE     9
HUMBERTO MAURO              8
GLAUBER ROCHA               8
GERALDO SARNO               8
OTTO GUERRA                 8
SERGIO ROIZENBLIT           8
FRANCISCO DE PAULA          8
ROGERIO SGANZERLA           7
Name: count, dtype: int64

E podemos exportar o arquivo para utilizar em outros apps.

## Exportando o resultado

Aqui, salvamos o dataframe como um CSV.

In [18]:
filmes_cols.to_csv('filmes_processados.csv', index=False, sep=';', encoding='utf-8-sig')